# 📦 TF-IDF Based Recommendation System (Scalable)

This notebook builds a content-based recommendation system using TF-IDF.

We will:
1. Load processed product data
2. Clean and prepare text
3. Perform TF-IDF vectorization
4. Build recommendation function
5. Evaluate results
6. Train production model (class-based)
7. Save model for deployment (Gradio)

💡 This notebook contains BOTH:
- Manual pipeline (for understanding)
- Class-based pipeline (for production)

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from src.models.tfidf_model import TFIDFRecommender

## Load Data

In [3]:
product_df = pd.read_csv("../data/processed/product_df.csv")

print("Shape:", product_df.shape)
product_df.head()

Shape: (30247, 7)


,asin,title,combined_text,avg_rating,rating_count,category_text,popularity
0,1118461304,NaN,clear lead innovation one thing book seemed ob...,4.600000,45,unknown,45
1,1906487049,NaN,hard find kim design experienced knitter like ...,4.666667,3,unknown,3
2,6040985461,NaN,look like original look like original keep min...,5.000000,1,unknown,1
3,7301113188,Tupperware Freezer Square Round Container Set ...,tupperware freezer square round container set ...,5.000000,1,unknown,1
4,7861850250,2 X Tupperware Pure &amp; Fresh Unique Covered...,x tupperware pure fresh unique covered cool cu...,3.000000,1,unknown,1


In [4]:
# Fill missing values
product_df["title"] = product_df["title"].fillna("")
product_df["combined_text"] = product_df["combined_text"].fillna("")

# Remove empty rows
product_df = product_df[product_df["combined_text"].str.strip() != ""]

print("Final shape:", product_df.shape)

Final shape: (30247, 7)


## 🔢 TF-IDF Vectorization

Convert text into numerical vectors.

In [5]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(product_df["combined_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (30247, 20000)


## 🔍 Recommendation Function (Query-Based)

Given a product title → return similar products.

In [6]:
def recommend(product_title, top_n=5):
    
    if product_title not in product_df["title"].values:
        return "❌ Product not found"
    
    idx = product_df[product_df["title"] == product_title].index[0]

    similarity_scores = cosine_similarity(
        tfidf_matrix[idx],
        tfidf_matrix
    ).flatten()

    top_indices = similarity_scores.argsort()[::-1][1:top_n+1]

    results = product_df.iloc[top_indices].copy()

    # Add business scoring
    results["score"] = (
        similarity_scores[top_indices]
    )

    return results.sort_values(by="score", ascending=False)[
        ["title", "asin", "avg_rating", "rating_count", "popularity", "score"]
    ]

## Testing Recommendation

In [7]:
sample_title = product_df["title"].iloc[100]

print("Query:", sample_title)
recommend(sample_title)

Query: Pfaltzgraff Cappuccino 8-Inch Enamel on Steel Burner Cover


,title,asin,avg_rating,rating_count,popularity,score
117,Pfaltzgraff Tea Rose 10-Inch Enamel on Steel B...,B0001GUCLK,3.00,2,2,0.732225
116,Pfaltzgraff Tea Rose 8-Inch Enamel on Steel Bu...,B0001GUCL0,4.75,4,4,0.719124
120,Pfaltzgraff Yorktowne 8-Inch Enamel on Steel B...,B0001GUCUG,3.00,2,2,0.716293
276,Gear Country Orchard 8-Inch Enamel on Steel Bu...,B0007A5O3G,5.00,2,2,0.681936
277,Gear Country Orchard 10-Inch Enamel on Steel B...,B0007A5O3Q,5.00,2,2,0.672294


In [8]:
popular_products = product_df.sort_values(by="popularity", ascending=False)

sample_title = popular_products["title"].iloc[0]

print("Popular product:", sample_title)
recommend(sample_title)

Popular product: General Electric MWF Refrigerator Water Filter


,title,asin,avg_rating,rating_count,popularity,score
26555,3-Pack Replacement General Electric MWF Refrig...,B015SO7U78,5.000000,6,6,0.795660
28941,MWF Refrigerator Water Filter,B01DKEHKSW,4.909091,11,11,0.732740
26549,Replacement General Electric PSS26SGPASS Refri...,B015SNWNZ8,5.000000,1,1,0.628877
26548,Replacement General Electric GSH25JSRFSS Refri...,B015SNWL64,3.750000,4,4,0.624876
26577,2-Pack Replacement General Electric GSHS6HGDBC...,B015SOVE4I,5.000000,3,3,0.615187


## 🏭 Production Model (Class-Based)

Now we train our reusable TFIDFRecommender class.

In [9]:
tfidf_model = TFIDFRecommender()

# IMPORTANT: pass full dataframe
tfidf_model.fit(product_df)

print("✅ Model trained")

✅ Model trained


In [10]:
tfidf_model.recommend(sample_title, top_n=5)

,title,asin,category_text,avg_rating,rating_count,popularity,score
26555,3-Pack Replacement General Electric MWF Refrig...,B015SO7U78,unknown,5.000000,6,6,0.795660
28941,MWF Refrigerator Water Filter,B01DKEHKSW,unknown,4.909091,11,11,0.732740
26549,Replacement General Electric PSS26SGPASS Refri...,B015SNWNZ8,unknown,5.000000,1,1,0.628877
26548,Replacement General Electric GSH25JSRFSS Refri...,B015SNWL64,unknown,3.750000,4,4,0.624876
26577,2-Pack Replacement General Electric GSHS6HGDBC...,B015SOVE4I,unknown,5.000000,3,3,0.615187


In [12]:
import pickle

os.makedirs("../artifacts", exist_ok=True)

with open("../artifacts/tfidf_model.pkl", "wb") as f:
    pickle.dump(tfidf_model, f)

print("✅ Model saved!")

✅ Model saved!
